In [ ]:
import pandas as pd
import numpy as np
import json
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.impute import SimpleImputer

articles = pd.read_csv("/content/articles_hm.csv")
customers = pd.read_csv("/content/customer_hm.csv")
transactions = pd.read_csv("/content/transactions_hm.csv")

articles.drop_duplicates(inplace=True)
customers.drop_duplicates(inplace=True)
transactions.drop_duplicates(inplace=True)

articles.columns = articles.columns.str.lower().str.strip()
customers.columns = customers.columns.str.lower().str.strip()
transactions.columns = transactions.columns.str.lower().str.strip()

required_transactions = ["t_dat", "customer_id", "article_id", "price"]
missing_transactions = [
    col for col in required_transactions
    if col not in transactions.columns
]

if missing_transactions:
    raise ValueError(f"Missing columns in transactions: {missing_transactions}")

if "customer_id" not in customers.columns:
    raise ValueError("Missing customer_id column in customers dataset")

if "article_id" not in articles.columns:
    raise ValueError("Missing article_id column in articles dataset")

if "detail_desc" in articles.columns:
    articles["detail_desc"] = articles["detail_desc"].fillna("No Description")

for col in articles.columns:
    if articles[col].dtype == "object":
        articles[col] = articles[col].fillna("UNKNOWN")

for col in ["club_member_status", "fashion_news_frequency"]:
    if col not in customers.columns:
        customers[col] = "UNKNOWN"
    customers[col] = customers[col].fillna("UNKNOWN")

if "age" not in customers.columns:
    customers["age"] = 0

customers["age"] = pd.to_numeric(customers["age"], errors="coerce")

age_imputer = SimpleImputer(strategy="median")
customers["age"] = age_imputer.fit_transform(customers[["age"]]).ravel()

transactions["price"] = pd.to_numeric(transactions["price"], errors="coerce")
transactions["price"] = transactions["price"].fillna(0)

transactions["t_dat"] = pd.to_datetime(transactions["t_dat"], errors="coerce")

transactions = transactions.dropna(
    subset=["t_dat", "customer_id", "article_id"]
)

articles["article_id"] = articles["article_id"].astype(str)
customers["customer_id"] = customers["customer_id"].astype(str)
transactions["article_id"] = transactions["article_id"].astype(str)
transactions["customer_id"] = transactions["customer_id"].astype(str)

transactions["year"] = transactions["t_dat"].dt.year
transactions["month"] = transactions["t_dat"].dt.month
transactions["day"] = transactions["t_dat"].dt.day
transactions["day_of_week"] = transactions["t_dat"].dt.dayofweek
transactions["week"] = transactions["t_dat"].dt.isocalendar().week.astype(int)
transactions["interaction"] = 1

categorical_article_cols = [
    "prod_name",
    "product_type_name",
    "product_group_name",
    "graphical_appearance_name",
    "colour_group_name",
    "perceived_colour_value_name",
    "perceived_colour_master_name",
    "department_name",
    "index_name",
    "index_group_name",
    "section_name",
    "garment_group_name"
]

categorical_customer_cols = [
    "club_member_status",
    "fashion_news_frequency"
]

article_encoders = {}
customer_encoders = {}

for col in categorical_article_cols:
    if col in articles.columns:
        le = LabelEncoder()
        articles[col] = le.fit_transform(articles[col].astype(str))
        article_encoders[col] = le

for col in categorical_customer_cols:
    if col in customers.columns:
        le = LabelEncoder()
        customers[col] = le.fit_transform(customers[col].astype(str))
        customer_encoders[col] = le

user_map = pd.DataFrame({
    "customer_id": transactions["customer_id"].unique()
})
user_map["user_idx"] = np.arange(len(user_map))

item_map = pd.DataFrame({
    "article_id": transactions["article_id"].unique()
})
item_map["item_idx"] = np.arange(len(item_map))

transactions = transactions.merge(
    user_map,
    on="customer_id",
    how="left"
)

transactions = transactions.merge(
    item_map,
    on="article_id",
    how="left"
)

scaler_age = MinMaxScaler()
scaler_price = MinMaxScaler()

customers[["age"]] = scaler_age.fit_transform(customers[["age"]])
transactions[["price"]] = scaler_price.fit_transform(transactions[["price"]])

merged_data = transactions.merge(
    customers,
    on="customer_id",
    how="left"
)

merged_data = merged_data.merge(
    articles,
    on="article_id",
    how="left"
)

merged_data = merged_data.dropna(
    subset=["user_idx", "item_idx", "t_dat"]
)

interaction_counts = merged_data.groupby(
    "customer_id"
)["article_id"].count().reset_index()

interaction_counts.columns = [
    "customer_id",
    "interaction_count"
]

merged_data = merged_data.merge(
    interaction_counts,
    on="customer_id",
    how="left"
)

article_popularity = merged_data.groupby(
    "article_id"
)["customer_id"].count().reset_index()

article_popularity.columns = [
    "article_id",
    "popularity_score"
]

merged_data = merged_data.merge(
    article_popularity,
    on="article_id",
    how="left"
)

final_data = merged_data.sort_values(by="t_dat")
final_data.reset_index(drop=True, inplace=True)

split_date = pd.Timestamp("2019-11-01")

train_data = final_data[
    final_data["t_dat"] < split_date
].copy()

valid_data = final_data[
    final_data["t_dat"] >= split_date
].copy()

if train_data.empty:
    raise ValueError("Train data is empty. Use a later split date.")

if valid_data.empty:
    raise ValueError("Validation data is empty. Use an earlier split date.")

final_data.to_csv(
    "/content/context_aware_recommendation_preprocessed.csv",
    index=False
)

train_data.to_csv(
    "/content/train_context_aware_recommendation.csv",
    index=False
)

valid_data.to_csv(
    "/content/valid_context_aware_recommendation.csv",
    index=False
)

user_map.to_csv(
    "/content/user_id_map.csv",
    index=False
)

item_map.to_csv(
    "/content/item_id_map.csv",
    index=False
)

summary = {
    "final_rows": int(len(final_data)),
    "train_rows": int(len(train_data)),
    "valid_rows": int(len(valid_data)),
    "unique_users": int(transactions["customer_id"].nunique()),
    "unique_items": int(transactions["article_id"].nunique()),
    "date_min": str(final_data["t_dat"].min()),
    "date_max": str(final_data["t_dat"].max())
}

with open("/content/preprocessing_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("Final data shape:", final_data.shape)
print("Train data shape:", train_data.shape)
print("Validation data shape:", valid_data.shape)
print(final_data.head())

final_data
final_data = final_data.fillna(0)
final_data.to_csv(
    "/content/final_context_aware_dataset.csv",
    index=False
)
from google.colab import files

files.download(
    "/content/final_context_aware_dataset.csv"
)
model_ready_columns = [
    "user_idx",
    "item_idx",
    "interaction",
    "price",
    "year",
    "month",
    "week",
    "day_of_week",
    "age",
    "interaction_count",
    "popularity_score",
    "club_member_status",
    "fashion_news_frequency",
    "product_type_name",
    "product_group_name",
    "colour_group_name",
    "section_name",
    "garment_group_name"
]

available_model_ready_columns = [
    col for col in model_ready_columns
    if col in final_data.columns
]

model_ready_dataset = final_data[
    available_model_ready_columns
].copy()

model_ready_dataset = model_ready_dataset.fillna(0)

model_ready_dataset = model_ready_dataset.drop_duplicates()

model_ready_dataset = model_ready_dataset.sample(
    n=min(100000, len(model_ready_dataset)),
    random_state=42
)

model_ready_dataset.to_csv(
    "/content/model_ready_recommendation_dataset.csv",
    index=False
)

print("Model-ready dataset shape:", model_ready_dataset.shape)
print(model_ready_dataset.head())

from google.colab import files

files.download(
    "/content/model_ready_recommendation_dataset.csv"
)
from google.colab import files

files.download("/content/model_ready_recommendation_dataset.csv")


Final data shape: (1040101, 44)
Train data shape: (893386, 44)
Validation data shape: (146715, 44)
       t_dat                                        customer_id article_id  \
0 2019-01-01  768212e65a56cfa7324c0c921bfdb78d635ab4c5612906...  567815005   
1 2019-01-01  e9d09a96df49a1b238cd341ff135ca668084a0879a9038...  673882002   
2 2019-01-01  8200d8003903fd1a568dbbebfe12111a717a9998768122...  641643002   
3 2019-01-01  fb364554cc836811a0c51da4f82b4dde2c8b6437be4a93...  480093001   
4 2019-01-01  ac53961282307b748fb51d0d019c1d4fc4a42331b330f1...  659452001   

      price  sales_channel_id  year  month  day  day_of_week  week  ...  \
0  0.026601                 2  2019      1    1            1     1  ...   
1  0.083149                 2  2019      1    1            1     1  ...   
2  0.026266                 2  2019      1    1            1     1  ...   
3  0.019574                 2  2019      1    1            1     1  ...   
4  0.200261                 2  2019      1    1          

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Model-ready dataset shape: (100000, 18)
        user_idx  item_idx  interaction     price  year  month  week  \
475313    222851      9996            1  0.016228  2019      6    24   
884809     73273     29908            1  0.149535  2019     10    43   
596727    119948     11029            1  0.025095  2019      7    28   
719122    261378      1845            1  0.042997  2019      8    34   
948460     88003     36588            1  0.133340  2019     11    48   

        day_of_week       age  interaction_count  popularity_score  \
475313            6  0.084337                  2                90   
884809            6  0.000000                 13                 8   
596727            6  0.566265                  2                54   
719122            2  0.000000                  4                33   
948460            3  0.277108                 18                 6   

        club_member_status  fashion_news_frequency  product_type_name  \
475313                 0.0       

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>